# Calcul manuel des scores F1 à partir des CSV de prédictions

Ce notebook recharge les CSV de prédictions des quatre configurations (Prompt 1 / Prompt 2, Baseline / Finetuné) et recalcule **manuellement** les F1-score (par classe et macro), en parallèle d’un calcul de vérification avec `sklearn`.

In [1]:
from pathlib import Path
from collections import Counter

import pandas as pd
from sklearn.metrics import f1_score

# Détection robuste du dossier results, quel que soit le dossier courant du kernel
_CWD = Path.cwd()
_candidates = [
    _CWD / "NLI4CT" / "results",      # si on lance le notebook depuis la racine du repo
    _CWD / "results",                  # si on lance depuis NLI4CT/
    _CWD.parent / "NLI4CT" / "results" # fallback supplémentaire
]

for _c in _candidates:
    if _c.exists():
        BASE_RESULTS = _c
        break
else:
    raise FileNotFoundError(f"Impossible de trouver le dossier results parmi : {[str(c) for c in _candidates]}")

In [2]:
def compute_manual_f1(y_true, y_pred):
    """Calcule manuellement F1 par classe et macro-F1.

    - On reproduit le comportement de `sklearn.metrics.f1_score` avec
      `average='macro', zero_division=0`.
    """
    y_true = list(y_true)
    y_pred = list(y_pred)

    labels = sorted(set(y_true) | set(y_pred))
    stats = {}

    for label in labels:
        tp = sum((yt == label) and (yp == label) for yt, yp in zip(y_true, y_pred))
        fp = sum((yt != label) and (yp == label) for yt, yp in zip(y_true, y_pred))
        fn = sum((yt == label) and (yp != label) for yt, yp in zip(y_true, y_pred))

        support = sum(yt == label for yt in y_true)

        # zero_division=0 pour précision / rappel
        precision = 0.0 if (tp + fp) == 0 else tp / (tp + fp)
        recall = 0.0 if (tp + fn) == 0 else tp / (tp + fn)
        f1 = 0.0 if (precision + recall) == 0 else 2 * precision * recall / (precision + recall)

        stats[label] = {
            "support": support,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }

    macro_f1 = sum(s["f1"] for s in stats.values()) / len(stats) if stats else 0.0
    return macro_f1, stats


def load_true_pred(csv_path: Path):
    df = pd.read_csv(csv_path)
    return df["true_label"], df["predicted_label"]

In [3]:
experiments = {
    "Prompt 1 Baseline": BASE_RESULTS / "Prompt 1" / "pred_bl_qwen7b_NLI4CT_prompt1.csv",
    "Prompt 2 Baseline": BASE_RESULTS / "Prompt 2" / "pred_bl_qwen7b_NLI4CT_prompt2.csv",
    "Prompt 1 Finetuné": BASE_RESULTS / "Prompt 1" / "pred_ft_qwen7b_NLI4CT_prompt1.csv",
    "Prompt 2 Finetuné": BASE_RESULTS / "Prompt 2" / "pred_ft_qwen7b_NLI4CT_prompt2.csv",
}

for name, path in experiments.items():
    print("=" * 80)
    print(name)
    print("CSV:", path)

    y_true, y_pred = load_true_pred(path)

    macro_f1_manual, stats = compute_manual_f1(y_true, y_pred)
    macro_f1_sklearn = f1_score(y_true, y_pred, average="macro", zero_division=0)

    print(f"Macro F1 (manuel)  : {macro_f1_manual:.4f}")
    print(f"Macro F1 (sklearn) : {macro_f1_sklearn:.4f}")
    print("\nDétail par label :")
    for label, s in stats.items():
        print(
            f"  - {label}: support={s['support']}, TP={s['tp']}, FP={s['fp']}, FN={s['fn']}, "
            f"precision={s['precision']:.3f}, recall={s['recall']:.3f}, f1={s['f1']:.3f}"
        )

    print()
print("=" * 80)
print("Terminé.")

Prompt 1 Baseline
CSV: NLI4CT/results/Prompt 1/pred_bl_qwen7b_NLI4CT_prompt1.csv


FileNotFoundError: [Errno 2] No such file or directory: 'NLI4CT/results/Prompt 1/pred_bl_qwen7b_NLI4CT_prompt1.csv'